# 06.2 - RAG - Evaluation

This notebook evaluates the RAG pipeline from notebook 08 using [RAGAS](https://docs.ragas.io), a framework for measuring retrieval-augmented generation quality without requiring hand-labeled answers for most metrics.

| Metric | What it measures | Ground truth needed? |
|---|---|---|
| **Faithfulness** | Are all claims in the answer supported by the retrieved context? | No |
| **Answer Relevancy** | Is the answer on-topic for the question? | No |
| **Context Precision** | Are the retrieved chunks relevant to the question? | No |
| **Context Recall** | Does the context cover everything needed to answer? | Yes |

The `output/medrxiv.db` Chroma database must exist (built by `scripts/07.2_medrxiv_knowledge_base.py`).

## 00. Dependencies

`ragas` and `datasets` are not in the main project dependencies.

In [ ]:
%pip install -q ragas datasets

## 01. Configuration

RAGAS uses an LLM as a judge for faithfulness and answer relevancy. Set `judge_model_id` to any LiteLLM-compatible model — a stronger model produces more reliable scores.

In [ ]:
import genscai.paths as paths

chroma_path = str(paths.output / "medrxiv.db")
collection_name = "articles_cosign_chunked_256"
rag_model_id = "ollama/qwen2.5:7b"  # model used to generate answers
judge_model_id = "ollama/qwen2.5:7b"  # model used by RAGAS to score outputs
n_results = 8

## 02. RAG Pipeline

Redefine the RAG functions from notebook 08 so this notebook is self-contained.

In [ ]:
import chromadb
from litellm import completion

client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_collection(collection_name)
print(f"{collection.count():,} chunks in collection")

SYSTEM_PROMPT = (
    "You are a research assistant specializing in infectious disease modeling. "
    "Answer the question using ONLY the provided research paper excerpts. "
    "Cite sources with [N] notation. "
    "If the excerpts do not contain enough information, say so explicitly."
)


def retrieve(query: str, n: int = n_results) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=n)
    chunks = results["documents"][0]
    metas = results["metadatas"][0]
    return [{"text": chunk, **meta} for chunk, meta in zip(chunks, metas)]


def format_context(hits: list[dict]) -> str:
    parts = []
    for i, h in enumerate(hits, 1):
        parts.append(f"[{i}] {h.get('title', 'Unknown')} ({str(h.get('date', ''))[:10]})\n    Excerpt: {h['text']}")
    return "\n\n".join(parts)


def generate(query: str, context: str) -> str:
    response = completion(
        model=rag_model_id,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Research excerpts:\n\n{context}\n\nQuestion: {query}"},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content


def rag(query: str) -> dict:
    hits = retrieve(query)
    context = format_context(hits)
    answer = generate(query, context)
    return {"answer": answer, "contexts": [h["text"] for h in hits]}

## 03. Evaluation Dataset

Ten domain-specific questions drawn from infectious disease modeling research. Five include reference answers that enable the Context Recall metric; the remaining five use only the no-reference metrics.

In [ ]:
eval_questions = [
    # --- questions with ground truth (enables context recall) ---
    "What approaches are used to model the spread of SARS-CoV-2?",
    "How is the basic reproduction number R0 estimated for respiratory diseases?",
    "What are the main limitations of SEIR models for real-world disease forecasting?",
    "What computational methods are used to fit epidemic models to surveillance data?",
    "How are non-pharmaceutical interventions modeled during pandemic response?",
    # --- questions without ground truth ---
    "What role do age-structured models play in pandemic planning?",
    "How do researchers model vaccine-induced immunity in compartmental models?",
    "How is spatial heterogeneity incorporated into infectious disease models?",
    "How do superspreading events affect disease transmission models?",
    "What is the role of behavioral change in epidemic model projections?",
]

# Reference answers for the first five questions (None = no ground truth)
ground_truths = [
    "SARS-CoV-2 spread is modeled using compartmental models (SEIR variants), agent-based models, and network models incorporating transmission parameters such as R0 and serial interval.",
    "R0 is estimated using exponential growth rate during the early epidemic phase, next-generation matrix methods, and Bayesian inference from case count data.",
    "SEIR model limitations include the homogeneous mixing assumption, constant parameters, absence of spatial structure, and difficulty capturing behavioral changes and population heterogeneity.",
    "Epidemic models are fitted to surveillance data using maximum likelihood estimation, MCMC sampling, and particle filtering calibrated to case counts, hospitalizations, and mortality.",
    "Non-pharmaceutical interventions are modeled by reducing the transmission rate parameter beta by intervention-specific factors, often estimated from mobility or contact survey data.",
    None,
    None,
    None,
    None,
    None,
]

## 04. Collect RAG Outputs

Run the full RAG pipeline on every evaluation question. This will make one LLM call per question.

In [ ]:
print("Running RAG pipeline on evaluation questions...")
rag_results = []
for i, q in enumerate(eval_questions):
    print(f"  [{i + 1:2d}/{len(eval_questions)}] {q[:65]}...")
    rag_results.append(rag(q))
print("Done.")

## 05. RAGAS Evaluation

Build a HuggingFace `Dataset` from the collected outputs and run RAGAS. Context Recall is included only for questions that have a reference answer.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, faithfulness
from ragas.llms import LiteLLMWrapper

# Configure RAGAS to use the same local model as the judge
judge_llm = LiteLLMWrapper(model=judge_model_id)
for metric in [faithfulness, answer_relevancy, context_precision]:
    metric.llm = judge_llm

data = {
    "question": eval_questions,
    "answer": [r["answer"] for r in rag_results],
    "contexts": [r["contexts"] for r in rag_results],
    "ground_truth": ground_truths,
}

ds = Dataset.from_dict(data)
metrics = [faithfulness, answer_relevancy, context_precision]

# Add context recall only for rows that have a ground truth
has_ground_truth = any(gt is not None for gt in ground_truths)
if has_ground_truth:
    from ragas.metrics import context_recall

    context_recall.llm = judge_llm
    metrics.append(context_recall)

ragas_result = evaluate(dataset=ds, metrics=metrics)
ragas_result

## 06. Results

Summarize mean scores and visualize per-question performance. Scores range from 0 to 1; values above 0.7 are generally considered acceptable.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

scores_df = ragas_result.to_pandas()
metric_cols = [c for c in scores_df.columns if c not in ("question", "answer", "contexts", "ground_truth")]

print("=== Mean RAGAS Scores ===")
print(scores_df[metric_cols].mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
means = scores_df[metric_cols].mean()
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
means.plot(kind="bar", ax=ax, color=colors[: len(means)])
ax.set_ylim(0, 1)
ax.set_title("RAGAS Metrics — Mean Scores")
ax.set_ylabel("Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.axhline(0.7, color="gray", linestyle="--", alpha=0.6, label="0.7 threshold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

short_q = [f"Q{i + 1}: {q[:50]}..." for i, q in enumerate(eval_questions)]
heat_data = scores_df[metric_cols].copy()
heat_data.index = short_q

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heat_data, vmin=0, vmax=1, annot=True, fmt=".2f", cmap="RdYlGn", ax=ax)
ax.set_title("RAGAS Scores per Question")
plt.tight_layout()
plt.show()

### Interpreting results

- **Low faithfulness**: The LLM is making claims not supported by retrieved chunks — try a stronger model or reduce `temperature`.
- **Low answer relevancy**: Answers are drifting off-topic — tighten the system prompt.
- **Low context precision**: The vector DB is returning noisy chunks — try increasing `n_results` and re-ranking, or switching to a science-specific embedding model (notebooks 04.2 / 04.3).
- **Low context recall**: The corpus doesn't cover the question well, or the embedding model is missing relevant papers — index more data or re-embed with a domain-specific model.